# 第1课：文档加载与分块

## 本节目标

理解 RAG 的第一步——将外部知识转化为可检索的文本片段。你将学会：

1. **文档加载**：从网页抓取、下载 PDF/HTML 文档
2. **文本解析**：从不同格式中提取纯文本
3. **文本分块（Chunking）**：将长文档切分为适合检索的小片段
4. **策略对比**：理解不同分块策略的优劣

> 为什么需要分块？因为 Embedding 模型有输入长度限制（通常 512 tokens），同时太长的文本会导致检索精度下降——"大海捞针"变成了"大海捞船"。

## 环境检查

In [ ]:
import sys
sys.path.insert(0, '..')

from config import config
from src.document_loader import DocumentLoader

print(f"项目根目录: {config.project_root}")
print(f"数据目录: {config.data_dir}")

## 1. 文档加载

`DocumentLoader` 负责获取知识源。它支持两种方式：

- **在线模式**：从 CMU 课程网站抓取讲义
- **离线模式**：使用内置的示例文档（10 个经典机器学习主题）

In [ ]:
# 创建加载器实例
loader = DocumentLoader()

# 加载文档：优先使用缓存，缓存不存在则从 CMU 网站抓取
# 如果无法访问 CMU 网站（离线环境），自动降级为内置示例文档
docs = loader.run()
print(f"加载了 {len(docs)} 篇文档")
for doc in docs[:3]:
    print(f"  - {doc['source']} ({doc['length']} 字符)")

## 2. 探索文档内容

看看每篇文档讲了什么。

In [ ]:
# 探索文档内容（如果上一步未执行，从缓存重新加载）
try:
    docs
except NameError:
    docs = loader.load_processed()

print(f"共 {len(docs)} 篇文档:\n")
for doc in docs:
    preview = doc['content'][:120].replace('\n', ' ')
    print(f"【{doc['source']}】({doc['length']} 字符)")
    print(f"  {preview}...\n")

## 3. 文本分块（Chunking）

这是 RAG 中最关键的工程决策之一。分块影响检索质量和 LLM 理解。

### 三种策略

| 策略 | 原理 | 优点 | 缺点 |
|------|------|------|------|
| **FixedTokenChunker** | 按固定 Token 数切片 | 简单可控 | 可能在句子中间截断 |
| **RecursiveCharChunker** | 按段落→句子→词的优先级递归切分 | 保持语义完整性 | 块大小不均匀 |
| **SemanticChunker** | 按句子边界切分，分组到阈值 | 保持句子完整 | 可能产生碎片 |

In [ ]:
from src.chunker import Chunker

# 使用默认策略（递归字符）创建分块器
chunker = Chunker(strategy="recursive_char", chunk_size=512, chunk_overlap=50)

# 对文档进行分块
chunks = chunker.chunk_documents(docs)

print(f"{len(docs)} 篇文档 → {len(chunks)} 个文本块")
print(f"\n前3个块的示例:")
for i, chunk in enumerate(chunks[:3]):
    print(f"\n--- 块 {i+1} ---")
    print(f"ID: {chunk['id']}")
    print(f"来源: {chunk['source']}")
    print(f"长度: {chunk['length']} 字符")
    print(f"内容预览: {chunk['content'][:200]}...")

## 4. 对比三种分块策略

不同策略产生的块在数量、平均长度、标准差上有什么差异？

In [ ]:
comparison = chunker.compare_strategies(docs)

print(f"{'策略':<20} {'块数':<8} {'平均长度':<12} {'最小长度':<10} {'最大长度':<10} {'标准差':<10}")
print("-" * 70)
for strategy, stats in comparison.items():
    print(f"{strategy:<20} {stats['num_chunks']:<8} {stats['avg_length']:<12.0f} "
          f"{stats['min_length']:<10} {stats['max_length']:<10} {stats['std_length']:<10.0f}")

## 5. 块重叠（Overlap）的作用

为什么需要重叠？想象一句关键的话刚好卡在分块边界上——前半个句子在一个块里，后半个在下一个块里。检索时可能两个块都匹配不上。

**重叠让相邻块共享部分内容，减少边界信息丢失。**

试试不同的 overlap 值：

In [ ]:
for overlap in [0, 50, 150]:
    test_chunker = Chunker(strategy="recursive_char", chunk_size=512, chunk_overlap=overlap)
    test_chunks = test_chunker.chunk_documents(docs[:3])
    avg_len = sum(c['length'] for c in test_chunks) / len(test_chunks) if test_chunks else 0
    print(f"overlap={overlap:3d}: {len(test_chunks)} 个块, 平均长度={avg_len:.0f} 字符")

## 本节小结

- 文档加载是 RAG 的入口，需要处理多种格式（PDF, HTML）
- 分块策略直接影响检索效果：`recursive_char` 通常是最平衡的选择
- 块大小和重叠是重要的超参数，需要根据文档类型和应用场景调优

**下一步**：在 02 号笔记本中，我们将把这些文本块转化为向量嵌入并构建索引。